# HW2 — Partitions, conditional entropy, and 1-WL on toy graphs

**Reading.** `handout.md` Q1–Q5.

**Goal.** Hand-derive $H(Y\mid\Pi)$ on a 6-node toy, code `cond_entropy`, implement one round of 1-WL refinement, and exhibit the $C_6$ vs $2K_3$ blind spot. Five assertion gates.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw2', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
# Reuse hbin from HW1's solution if exported, else re-define inline.
from math import log2
def hbin(p: float) -> float:
    return 0.0 if (p <= 0 or p >= 1) else -(p*log2(p) + (1-p)*log2(1-p))


## Q1 — Hand-compute on a 6-node toy

$y = (0,0,1,1,1,0)$, $\Pi = \{\{0,1,2\}, \{3,4\}, \{5\}\}$.

By hand: $q = (1/2, 1/3, 1/6)$, $e = (1/3, 0, 0)$, $\varepsilon(\Pi) = 1/6 \approx 0.1667$, $H(Y\mid\Pi) = \tfrac{1}{2} H_\mathrm{bin}(1/3) \approx 0.4591$.

In [ ]:
y = np.array([0,0,1,1,1,0])
Pi = [np.array([0,1,2]), np.array([3,4]), np.array([5])]
n = sum(len(c) for c in Pi)
q = [len(c)/n for c in Pi]
e = []
for c in Pi:
    lbls = y[c]
    vals, counts = np.unique(lbls, return_counts=True)
    e.append(1 - counts.max()/len(c))
eps = sum(qi*ei for qi, ei in zip(q, e))
H = sum(qi*hbin(ei) for qi, ei in zip(q, e))
print(f'q={q}, e={e}, eps={eps:.4f}, H(Y|Pi)={H:.4f}')


**Gate Q1.** Hand and code must agree to $10^{-12}$.

In [ ]:
assert abs(q[0]-1/2) < 1e-12 and abs(q[1]-1/3) < 1e-12 and abs(q[2]-1/6) < 1e-12
assert abs(e[0]-1/3) < 1e-12 and e[1]==0 and e[2]==0
assert abs(eps - 1/6) < 1e-12, f'eps {eps} ≠ 1/6'
assert abs(H - 0.5*hbin(1/3)) < 1e-12, f'H {H} ≠ 0.5·hbin(1/3)'
print(f'[GATE OK] Q1: q,e,eps,H match hand computation; H={H:.6f}')


In [ ]:
reflect.log('Q2.hw2.Q1_hand', f'H(Y|Pi) on 6-node toy ≈ {H:.4f}', 'HIGH')


## Q2 — `cond_entropy(partition, labels)`

**Concept.** Sum cell-wise $q_C \cdot H_\mathrm{bin}(e_C)$. Edge cases: every cell singleton ⇒ $H = 0$. Single all-of-$\{0,\dots,n-1\}$ cell ⇒ $H = H_\mathrm{bin}(\varepsilon)$ (global Bayes error).

In [ ]:
def cond_entropy(partition, labels):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — partition-entropy vs label-entropy.** $H(Y\mid\Pi)$ is NOT $H(Y)$; it's $H$ of the *Bayes error inside each cell*. Singletons drive $H$ to 0; the trivial 1-cell partition recovers $H_\mathrm{bin}(\varepsilon_{\mathrm{global}})$.

In [ ]:
# Singleton-cell partition: every cell has e=0 ⇒ H = 0.
sing = [np.array([i]) for i in range(len(y))]
H_sing = cond_entropy(sing, y)
# Trivial 1-cell partition: H = hbin(global Bayes error).
triv = [np.arange(len(y))]
H_triv = cond_entropy(triv, y)
_, counts = np.unique(y, return_counts=True)
eps_global = 1 - counts.max()/len(y)
print(f'H_sing={H_sing:.6f} H_triv={H_triv:.6f} hbin(eps_global)={hbin(eps_global):.6f}')
H_toy = cond_entropy(Pi, y)
print(f'H_toy={H_toy:.6f}')


**Gate Q2.** Singletons ⇒ 0; trivial ⇒ $H_\mathrm{bin}(\varepsilon)$; toy matches Q1.

In [ ]:
assert H_sing == 0.0, f'Q2: singleton H={H_sing} should be 0'
assert abs(H_triv - hbin(eps_global)) < 1e-12, f'Q2: triv {H_triv} ≠ hbin({eps_global})'
assert abs(H_toy - H) < 1e-12, f'Q2: toy {H_toy} ≠ hand {H}'
print(f'[GATE OK] Q2: cond_entropy matches edges and toy (H_toy={H_toy:.4f})')


In [ ]:
reflect.log('Q2.hw2.Q2_cond_entropy', 'cond_entropy correct on singleton, trivial, toy', 'HIGH')


## Q3+Q4 — One-round 1-WL refinement and the $C_6$ vs $2K_3$ blind spot

**Concept.** $\mathrm{color}_{t+1}(u) = \mathrm{hash}(\mathrm{color}_t(u), \{\!\!\{\mathrm{color}_t(v) : v \sim u\}\!\!\})$. Rename signatures densely each round. $C_6$ and $2K_3$ are both 2-regular and indistinguishable by 1-WL.

In [ ]:
def wl_step(edge_index, n, colors):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — 1-WL vs degree.** A 1-WL round subsumes degree (multiset size). Two regular graphs of the same degree (e.g. $C_6$ and $2K_3$) cannot be told apart even after stability.

In [ ]:
def edges_C6():
    pairs = [(i, (i+1) % 6) for i in range(6)]
    e = []
    for u, v in pairs:
        e += [(u, v), (v, u)]
    return np.array(e).T

def edges_2K3():
    pairs = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5)]
    e = []
    for u, v in pairs:
        e += [(u, v), (v, u)]
    return np.array(e).T

def wl_stabilise(edge_index, n, rounds=10):
    colors = np.zeros(n, dtype=int)
    for _ in range(rounds):
        new = wl_step(edge_index, n, colors)
        if np.array_equal(new, colors):
            break
        colors = new
    return colors

c6 = wl_stabilise(edges_C6(), 6)
k3 = wl_stabilise(edges_2K3(), 6)
def multiset(arr):
    _, cnts = np.unique(arr, return_counts=True)
    return tuple(sorted(cnts.tolist()))
print(f'C6 stable colors: {c6}  multiset {multiset(c6)}')
print(f'2K3 stable colors: {k3}  multiset {multiset(k3)}')


**Gate Q3+Q4.** $C_6$ and $2K_3$ stable multisets coincide; both equal $(6,)$.

In [ ]:
assert multiset(c6) == multiset(k3) == (6,), f'1-WL multisets {multiset(c6)} vs {multiset(k3)} — blind-spot test failed'
# Also check wl_step is non-trivial on the path P_3: endpoints split from middle.
p3 = np.array([[0,1,1,2],[1,0,2,1]])
col = np.zeros(3, dtype=int)
for _ in range(3):
    col = wl_step(p3, 3, col)
assert len(set(col.tolist())) == 2, f'P3 should split into 2 colour classes; got {col}'
assert col[0] == col[2] and col[0] != col[1], f'P3 endpoints {col[0]},{col[2]} vs middle {col[1]}'
print(f'[GATE OK] Q3+Q4: C6 and 2K3 indistinguishable by 1-WL; P3 endpoints split from middle')


In [ ]:
reflect.log('Q2.hw2.Q3Q4_wl_blindspot',
            'C6 vs 2K3 share stable-colour multiset (6,) under 1-WL; P3 separates endpoints from middle',
            'HIGH')


## Q5 — Writeup & calibration

`writeup.md` should mirror §Q1–§Q4. Below we roll up the reflection log.

In [ ]:
reflect.log('Q2.hw2.Q5_writeup',
            'Writeup mirrors hand-comp, cond_entropy, wl_step, blind-spot demonstration',
            'MEDIUM')
print('HW2 reflection log:')
for r in reflect.dump():
    if 'hw2' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW2.**